In [ ]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null # Instalar SDK java 17

In [ ]:
!wget -q https://downloads.apache.org/spark/spark-4.0.2/spark-4.0.2-bin-hadoop3.tgz # Descargar Spark 4.0.1

In [ ]:
!tar xf spark-4.0.2-bin-hadoop3.tgz # Descomprimir la version de Spark

In [ ]:
!pip install -q findspark # Instalar la librería findspark

In [ ]:
!pip install -q pyspark # Instalar pyspark

# Catalyst Optimizer

La forma más fácil de escribir aplicaciones de procesamiento de datos eficientes es no preocuparse por ello y optimizar automáticamente sus aplicaciones de procesamiento de datos.

El primer paso de Catalyst es crear un lógica a partir de un objetivo DataFrame o del árbol de sintaxis abstracta de la consulta SQL analizada.


In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
data = spark.read.parquet('/content/sample_data/vuelos.parquet')

In [ ]:
data.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [ ]:
data.show()

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

In [ ]:
from pyspark.sql.functions import col

In [ ]:
nuevo_df = (data.filter(col('MONTH').isin(6,7,8))
            .withColumn('dis_tiempo_aire', col('DISTANCE') / col('AIR_TIME'))
).select(
    col('AIRLINE'),
    col('dis_tiempo_aire')
).where(col('AIRLINE').isin('AA', 'DL', 'AS'))

In [ ]:
nuevo_df.explain(True)

== Parsed Logical Plan ==
'Filter 'in('AIRLINE, AA, DL, AS)
+- Project [AIRLINE#4, dis_tiempo_aire#126]
   +- Project [YEAR#0, MONTH#1, DAY#2, DAY_OF_WEEK#3, AIRLINE#4, FLIGHT_NUMBER#5, TAIL_NUMBER#6, ORIGIN_AIRPORT#7, DESTINATION_AIRPORT#8, SCHEDULED_DEPARTURE#9, DEPARTURE_TIME#10, DEPARTURE_DELAY#11, TAXI_OUT#12, WHEELS_OFF#13, SCHEDULED_TIME#14, ELAPSED_TIME#15, AIR_TIME#16, DISTANCE#17, WHEELS_ON#18, TAXI_IN#19, SCHEDULED_ARRIVAL#20, ARRIVAL_TIME#21, ARRIVAL_DELAY#22, DIVERTED#23, CANCELLED#24, ... 7 more fields]
      +- Filter MONTH#1 IN (6,7,8)
         +- Relation [YEAR#0,MONTH#1,DAY#2,DAY_OF_WEEK#3,AIRLINE#4,FLIGHT_NUMBER#5,TAIL_NUMBER#6,ORIGIN_AIRPORT#7,DESTINATION_AIRPORT#8,SCHEDULED_DEPARTURE#9,DEPARTURE_TIME#10,DEPARTURE_DELAY#11,TAXI_OUT#12,WHEELS_OFF#13,SCHEDULED_TIME#14,ELAPSED_TIME#15,AIR_TIME#16,DISTANCE#17,WHEELS_ON#18,TAXI_IN#19,SCHEDULED_ARRIVAL#20,ARRIVAL_TIME#21,ARRIVAL_DELAY#22,DIVERTED#23,CANCELLED#24,... 6 more fields] parquet

== Analyzed Logical Plan ==
AIRL